### Generating Embeddings from Images
OpenAI CLIP is open-sourced and we can use it for image to text, text to image, image-to-image and text to text search.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in your directory
load_dotenv()

# Retrieve the API key from the environment
api_key = os.getenv("OPENAI_API_KEY")

# Verify if the key was loaded successfully
if api_key:
    print(f"✓ API Key loaded successfully. Starts with: {api_key[:7]}...")
else:
    print("✗ API Key not found. Please check your .env file.")

✓ API Key loaded successfully. Starts with: sk-proj...


In [4]:
#!pip install sentence_transformers

In [12]:
from huggingface_hub import InferenceClient

In [13]:
from sentence_transformers import SentenceTransformer
from PIL import Image

In [14]:
# Read token from environment variable
token = os.getenv("HUGGINGFACE_ACCESS_TOKEN")
# Initialize client with token
#client = InferenceClient(token=token)

In [19]:
import torch
from transformers import CLIPModel, CLIPProcessor

# 1. Load the processor and model directly from transformers
model_id = 'openai/clip-vit-base-patch32'
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id)

# 2. Process image and text inputs
image = Image.open('fox_in_forest.jpg')
texts = ['A fox in a forest', 'A cat on a table', 'A picture of London at night']

# Prepare inputs for the model
inputs = processor(
    text=texts, 
    images=image, 
    return_tensors="pt", 
    padding=True
)

# 3. Forward pass (generate embeddings)
with torch.no_grad():
    outputs = model(**inputs)

# Image and text embeddings
image_emb = outputs.image_embeds  # Shape: [1, 512]
text_emb = outputs.text_embeds    # Shape: [3, 512]

print("Image embedding shape:", image_emb.shape)
print("Text embeddings shape:", text_emb.shape)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Image embedding shape: torch.Size([1, 512])
Text embeddings shape: torch.Size([3, 512])


In [20]:
print(len(image_emb))

1


In [21]:
print(len(text_emb))
print(len(text_emb[0]))

3
512


In [23]:
#Compute cosine similarities 

# 1. Load the SBERT-compatible CLIP model wrapper
model = SentenceTransformer('sentence-transformers/clip-ViT-B-32')

# 2. Encode the image (returns a 1D or 2D PyTorch tensor/numpy array)
image_emb = model.encode(Image.open('fox_in_forest.jpg'))

# 3. Encode the text descriptions
text_emb = model.encode([
    'A fox in a forest', 
    'A cat on a table', 
    'A picture of London at night'
])

# 4. Compute cosine similarities (Image vs Texts)
similarity_scores = model.similarity(image_emb, text_emb)

print("Similarity Scores:")
print(similarity_scores)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /home/maulik/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity Scores:
tensor([[0.3047, 0.1946, 0.1749]])
